In [ ]:
# networks.py
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ---------------- Attention ---------------- #
# networks.py - Improved Attention
class SelfAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.scale = dim ** 0.5

    def forward(self, x):
        Q = self.q(x) # (batch, 256)
        K = self.k(x) # (batch, 256)
        V = self.v(x) # (batch, 256)
    
        # Feature-wise attention: calculate weight for each feature independently
        # This matches the "scaling" intent of the base paper [cite: 9, 42]
        attn_scores = torch.softmax((Q * K) / self.scale, dim=-1) 
        return attn_scores * V


# ---------------- Actor ---------------- #
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 256)
        self.attn = SelfAttention(256)
        self.fc2 = nn.Linear(256, 128)
        self.out = nn.Linear(128, action_dim)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = self.attn(x)
        x = F.relu(self.fc2(x))
        return torch.tanh(self.out(x))

# ---------------- Critic ---------------- #
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)
